# 01 · Generación e ingesta — capa Bronze

**Pregunta que prepara este notebook:** ¿con qué datos podemos responder cuáles líneas de pedido se devuelven?

## Qué pasa en este notebook

El proyecto no tiene una fuente real, así que los datos se **generan sintéticamente** para una tienda online de moda rápida. El resultado son cuatro tablas en Bronze, la capa donde los datos se guardan tal como llegan del sistema origen, con sus errores incluidos:

| Tabla Bronze | Qué contiene | Sistema origen simulado |
|---|---|---|
| `clientes` | Quién compra: ciudad, segmento (Nuevo, Recurrente, VIP) y fecha de registro | CRM |
| `productos` | Catálogo: categoría, talla, color y precio de lista | ERP de inventario |
| `lineas_pedido` | Una fila por prenda comprada: cliente, producto, fecha, cantidad, descuento y precio pagado | Plataforma de pedidos |
| `devoluciones` | Qué líneas se devolvieron y cuándo. **Llega aparte, después de la compra** | Portal de devoluciones |

## Decisiones que conviene entender

1. **La etiqueta no viaja con la línea.** En un negocio real, al momento de la compra nadie sabe si la prenda se devolverá. Por eso las devoluciones son una tabla separada: la información llega días después, y en `02_Delta_Lake` un `MERGE INTO` la aplica sobre las líneas ya escritas.
2. **Los datos nacen sucios a propósito.** Hay duplicados, claves nulas, cantidades en cero, tallas escritas de varias formas (`M`, `m`, `Medium`), colores nulos y precios como texto con `$` o con coma decimal. Así la capa Silver tiene un trabajo real que hacer.
3. **Hay señal, pero no una regla exacta.** La probabilidad de devolución sube con las tallas extremas, los vestidos y el calzado, los descuentos altos y los clientes nuevos, y baja con los clientes VIP. Luego se hace un sorteo, así que el resultado tiene ruido: un modelo puede aprender patrones, pero no acertar el 100 %.
4. **Aleatoriedad reproducible con `F.hash`.** El curso usa `F.rand(SEED)`. Aquí se usa un número derivado de `F.hash` sobre el identificador de la línea, porque el mismo sorteo alimenta dos tablas (las líneas y las devoluciones) y `F.rand` puede cambiar si Spark reordena las filas. Con `F.hash`, el valor depende solo del identificador y de la semilla, así que las dos tablas salen siempre consistentes.
5. **Una devolución puede tardar hasta 35 días, pero solo cuenta si llega en 30.** Eso mete un detalle realista: parte de las devoluciones llegan fuera de la ventana y la etiqueta las deja en 0.

**En esta celda** se hereda la configuración compartida con `%run`: catálogo, variables, `tbl()` y `escribir_delta()`.

In [0]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importan las funciones y tipos de PySpark (cada notebook importa lo que necesita, porque `%run` comparte variables pero no imports) y se define `u()`, el generador de números uniformes entre 0 y 1 basado en `F.hash`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast


def u(col, salt):
    """Numero 'aleatorio' uniforme en [0, 1) que depende SOLO del valor de `col` y de la semilla.
    Usa F.hash en vez de F.rand para que el mismo sorteo sea identico aunque Spark reordene las filas."""
    return F.pmod(F.hash(F.col(col), F.lit(SEED + salt)), F.lit(1000000)) / F.lit(1000000.0)

**En esta celda** se generan las dos dimensiones **limpias** (clientes y productos) con el mismo patrón del curso: `spark.range` para crear las filas y `F.element_at` sobre un arreglo con `id % N` para repartir las categorías de forma cíclica. Se llaman `_limpio` porque más abajo se les inyecta la suciedad; esta versión limpia sirve para calcular la probabilidad real de devolución.

In [0]:
categorias = ["Vestidos", "Camisetas", "Jeans", "Chaquetas", "Faldas", "Calzado"]
tallas     = ["XS", "S", "M", "L", "XL"]
colores    = ["Negro", "Blanco", "Azul", "Rojo", "Verde", "Beige", "Rosa"]
ciudades   = ["Bogota", "Medellin", "Cali", "Barranquilla", "Bucaramanga", "Cartagena"]
segmentos  = ["Nuevo", "Recurrente", "VIP"]
descuentos = [0, 0, 0, 10, 20, 30, 40, 50]
motivos    = ["Talla incorrecta", "No coincide con la foto", "Defecto de calidad", "Cambio de opinion"]

# TABLA 1: clientes — dimension principal (CRM)
df_clientes_limpio = (
    spark.range(1, NUM_CLIENTES + 1)
    .withColumn("cliente_id", F.col("id").cast(IntegerType()))
    .withColumn("nombre",   F.concat(F.lit("Cliente_"), F.lpad(F.col("id").cast(StringType()), 6, "0")))
    .withColumn("email",    F.concat(F.lit("cliente"), F.col("id").cast(StringType()), F.lit("@correo-demo.com")))
    .withColumn("ciudad",   F.element_at(F.array(*[F.lit(c) for c in ciudades]),  (F.col("id") % 6 + 1).cast(IntegerType())))
    .withColumn("segmento", F.element_at(F.array(*[F.lit(s) for s in segmentos]), (F.col("id") % 3 + 1).cast(IntegerType())))
    .withColumn("fecha_registro", F.date_add(F.lit("2022-01-01").cast(DateType()), (F.col("id") * 3 % 1000).cast(IntegerType())))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("_source_system", F.lit("CRM_MODA"))
    .drop("id")
)

# TABLA 2: productos — catalogo (ERP). Cada fila es un SKU: una prenda en una talla y un color.
df_productos_limpio = (
    spark.range(1, NUM_PRODUCTOS + 1)
    .withColumn("producto_id", F.col("id").cast(IntegerType()))
    .withColumn("categoria", F.element_at(F.array(*[F.lit(c) for c in categorias]), (F.col("id") % 6 + 1).cast(IntegerType())))
    .withColumn("talla",     F.element_at(F.array(*[F.lit(t) for t in tallas]),     (F.col("id") % 5 + 1).cast(IntegerType())))
    .withColumn("color",     F.element_at(F.array(*[F.lit(c) for c in colores]),    (F.col("id") % 7 + 1).cast(IntegerType())))
    .withColumn("nombre",    F.concat(F.col("categoria"), F.lit("_"), F.lpad(F.col("id").cast(StringType()), 4, "0")))
    .withColumn("precio_lista", (F.lit(30000) + (F.col("id") * 7919 % 220000)).cast(IntegerType()))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("_source_system", F.lit("ERP_INVENTARIO"))
    .drop("id")
)
print(f"Clientes (limpio): {df_clientes_limpio.count():,}   Productos (limpio): {df_productos_limpio.count():,}")

**En esta celda** se generan las líneas de pedido y, sobre ellas, la **probabilidad real de devolución** (columnas con prefijo `_`, que no se guardan). Primero se asignan cliente, producto, fecha, cantidad y descuento; luego se une con las dimensiones limpias para saber la talla, la categoría y el segmento de cada línea; después se suman los ajustes de riesgo y, por último, se hace el sorteo con `u()`.

**Cómo se controla la tasa:** `AJUSTE_CENTRO` resta el promedio de los ajustes para que la probabilidad media quede cerca de la tasa base del widget (22 %). La tasa efectiva final es menor (alrededor de 19 %), porque las devoluciones que tardan más de 30 días no cuentan.

In [0]:
AJUSTE_CENTRO = 0.163   # promedio de los ajustes de riesgo; se resta para que la probabilidad media ~ TASA_DEVOLUCION

df_lineas_base = (
    spark.range(1, NUM_LINEAS + 1)
    .withColumn("linea_id",   F.col("id").cast(IntegerType()))
    .withColumn("pedido_id",  F.floor((F.col("id") + 1) / 2).cast(IntegerType()))
    .withColumn("cliente_id",  (F.floor(u("linea_id", 1) * NUM_CLIENTES) + 1).cast(IntegerType()))
    .withColumn("producto_id", (F.floor(u("linea_id", 2) * NUM_PRODUCTOS) + 1).cast(IntegerType()))
    .withColumn("fecha", F.date_add(F.lit(FECHA_INICIO).cast(DateType()), F.floor(u("linea_id", 3) * 365).cast(IntegerType())))
    .withColumn("cantidad", F.when(u("linea_id", 4) < 0.85, F.lit(1)).otherwise(F.lit(2)))
    .withColumn("descuento_pct", F.element_at(F.array(*[F.lit(d) for d in descuentos]), (F.floor(u("linea_id", 5) * 8) + 1).cast(IntegerType())))
    .drop("id")
)

df_lineas_latente = (
    df_lineas_base
    .join(broadcast(df_productos_limpio.select("producto_id", "categoria", "talla", "precio_lista")), on="producto_id", how="inner")
    .join(broadcast(df_clientes_limpio.select("cliente_id", "segmento")), on="cliente_id", how="inner")
    # Ajustes de riesgo: cada factor suma probabilidad de devolucion
    .withColumn("_ajuste_talla",     F.when(F.col("talla").isin("XS", "XL"), F.lit(0.15)).otherwise(F.lit(0.0)))
    .withColumn("_ajuste_categoria", F.when(F.col("categoria").isin("Vestidos", "Calzado"), F.lit(0.12)).otherwise(F.lit(0.0)))
    .withColumn("_ajuste_descuento", F.col("descuento_pct") * F.lit(0.003))
    .withColumn("_ajuste_segmento",  F.when(F.col("segmento") == "Nuevo", F.lit(0.10))
                                      .when(F.col("segmento") == "VIP", F.lit(-0.08))
                                      .otherwise(F.lit(0.0)))
    .withColumn("_p_devolucion", F.greatest(F.lit(0.02), F.least(F.lit(0.85),
        F.lit(TASA_DEVOLUCION - AJUSTE_CENTRO) + F.col("_ajuste_talla") + F.col("_ajuste_categoria")
        + F.col("_ajuste_descuento") + F.col("_ajuste_segmento"))))
    # Sorteo: la linea se devuelve si el numero uniforme cae por debajo de su probabilidad
    .withColumn("_devuelta", u("linea_id", 6) < F.col("_p_devolucion"))
    .withColumn("_dias_hasta_devolucion", (F.floor(u("linea_id", 7) * MAX_DIAS_HASTA_DEVOLUCION) + 1).cast(IntegerType()))
    .withColumn("_precio_pagado", F.round(F.col("precio_lista") * F.col("cantidad") * (F.lit(1.0) - F.col("descuento_pct") / F.lit(100.0)), 2))
)

resumen = df_lineas_latente.agg(
    F.count("*").alias("lineas"),
    F.avg(F.col("_p_devolucion")).alias("p_media"),
    F.avg(F.col("_devuelta").cast("int")).alias("tasa_devuelta"),
    F.avg((F.col("_devuelta") & (F.col("_dias_hasta_devolucion") <= VENTANA_DEVOLUCION_DIAS)).cast("int")).alias("tasa_efectiva_30d"),
).first()
print(f"Lineas generadas             : {resumen['lineas']:,}")
print(f"Probabilidad media de devol. : {resumen['p_media']:.1%}")
print(f"Lineas que se devuelven      : {resumen['tasa_devuelta']:.1%}")
print(f"Tasa efectiva (<= {VENTANA_DEVOLUCION_DIAS} dias)     : {resumen['tasa_efectiva_30d']:.1%}")

**En esta celda** se construye la tabla `devoluciones` a partir del mismo sorteo: solo las líneas devueltas, con la fecha en que llegó el registro (fecha de compra más los días de demora) y un motivo. Es el "segundo lote" que más adelante se aplica con `MERGE INTO`.

In [0]:
df_devoluciones = (
    df_lineas_latente
    .filter(F.col("_devuelta"))
    .select(
        F.col("linea_id"),
        F.date_add(F.col("fecha"), F.col("_dias_hasta_devolucion")).alias("fecha_devolucion"),
        F.element_at(F.array(*[F.lit(m) for m in motivos]), (F.floor(u("linea_id", 8) * 4) + 1).cast(IntegerType())).alias("motivo"),
    )
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("_source_system", F.lit("PORTAL_DEVOLUCIONES"))
)
print(f"Devoluciones generadas: {df_devoluciones.count():,}")
display(df_devoluciones.groupBy("motivo").count().orderBy(F.desc("count")))

**En esta celda** se inyecta la **suciedad** en las tres tablas que llegan a Bronze. Cada error tiene una razón de ser, porque es lo que Silver va a tener que arreglar:

- **Clientes:** 3 % de ciudades nulas.
- **Productos:** tallas escritas de tres formas (`M`, `m`, `Medium`), 5 % de colores nulos y 4 % de precios como texto con prefijo `$`.
- **Líneas de pedido:** 1 % de `cliente_id` nulo, 1 % de cantidad en 0, 3 % de precios con coma decimal, el precio guardado como texto, y alrededor de 1,5 % de filas duplicadas.

In [0]:
# --- Clientes: ciudad nula en ~3% ---
df_clientes = df_clientes_limpio.withColumn(
    "ciudad", F.when(u("cliente_id", 12) < 0.03, F.lit(None).cast(StringType())).otherwise(F.col("ciudad")))

# --- Productos: tallas con formatos distintos, colores nulos, precio como texto ---
df_productos = (
    df_productos_limpio
    .withColumn("talla",
        F.when(u("producto_id", 20) < 0.12, F.lower(F.col("talla")))                       # 'm'
         .when(u("producto_id", 20) < 0.20,                                                 # 'Medium'
               F.when(F.col("talla") == "XS", "Extra Small").when(F.col("talla") == "S", "Small")
                .when(F.col("talla") == "M", "Medium").when(F.col("talla") == "L", "Large")
                .otherwise("Extra Large"))
         .otherwise(F.col("talla")))
    .withColumn("color", F.when(u("producto_id", 21) < 0.05, F.lit(None).cast(StringType())).otherwise(F.col("color")))
    .withColumn("precio_lista",
        F.when(u("producto_id", 22) < 0.04, F.concat(F.lit("$"), F.col("precio_lista").cast(StringType())))
         .otherwise(F.col("precio_lista").cast(StringType())))
)

# --- Lineas de pedido: claves nulas, cantidad 0, precio como texto (a veces con coma), duplicados ---
df_lineas_sucias = (
    df_lineas_latente
    .withColumn("cliente_id", F.when(u("linea_id", 9) < 0.01, F.lit(None).cast(IntegerType())).otherwise(F.col("cliente_id")))
    .withColumn("cantidad",   F.when(u("linea_id", 11) < 0.01, F.lit(0)).otherwise(F.col("cantidad")))
    .withColumn("precio_pagado",
        F.when(u("linea_id", 13) < 0.03, F.regexp_replace(F.col("_precio_pagado").cast(StringType()), r"\.", ","))
         .otherwise(F.col("_precio_pagado").cast(StringType())))
    .withColumn("mes", F.date_format(F.col("fecha"), "yyyy-MM"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("_source_system", F.lit("PLATAFORMA_PEDIDOS"))
    .select("linea_id", "pedido_id", "cliente_id", "producto_id", "fecha", "mes", "cantidad",
            "descuento_pct", "precio_pagado", "ingested_at", "_source_system")
)
df_duplicadas = df_lineas_sucias.sample(withReplacement=False, fraction=0.015, seed=SEED)
df_lineas_pedido = df_lineas_sucias.unionByName(df_duplicadas)
print("Suciedad inyectada en clientes, productos y lineas de pedido.")

**En esta celda** se escriben las cuatro tablas a Bronze con `escribir_delta()` (activando `optimize_write`). Las líneas se particionan por `mes` (12 valores, baja cardinalidad y columna típica de filtro), no por `fecha`, para evitar cientos de carpetas diminutas, que es el "small file problem" que vimos en el curso.

In [0]:
escribir_delta(df_clientes,      SCHEMA_BRONZE, "clientes",      "Clientes - dimension principal (CRM)",               optimize_write=True)
escribir_delta(df_productos,     SCHEMA_BRONZE, "productos",     "Catalogo de productos, un SKU por fila (ERP)",       optimize_write=True)
escribir_delta(df_lineas_pedido, SCHEMA_BRONZE, "lineas_pedido", "Lineas de pedido tal como llegan, con errores",      partition_col="mes", optimize_write=True)
escribir_delta(df_devoluciones,  SCHEMA_BRONZE, "devoluciones",  "Registros de devolucion, llegan despues de la compra", optimize_write=True)

print("\nBronze listo. Siguiente: 01_Exploracion_y_Transformacion.")

**En esta celda** se verifica Bronze: cuántas filas tiene cada tabla y cuánta suciedad quedó adentro, para comprobar que los errores inyectados realmente están ahí antes de limpiarlos.

In [0]:
print("=" * 60)
print("VERIFICACION BRONZE")
print("=" * 60)
for tabla in ["clientes", "productos", "lineas_pedido", "devoluciones"]:
    print(f"  {tabla:<16} {spark.table(tbl(SCHEMA_BRONZE, tabla)).count():>10,} registros")

l = spark.table(tbl(SCHEMA_BRONZE, "lineas_pedido"))
p = spark.table(tbl(SCHEMA_BRONZE, "productos"))
n_dup      = l.count() - l.select("linea_id").distinct().count()
n_cli_nulo = l.filter("cliente_id IS NULL").count()
n_cant_0   = l.filter("cantidad <= 0").count()
n_coma     = l.filter("precio_pagado LIKE '%,%'").count()
n_color    = p.filter("color IS NULL").count()
n_dolar    = p.filter("precio_lista LIKE '$%'").count()
print("\nSuciedad presente:")
print(f"  lineas duplicadas por linea_id : {n_dup:,}")
print(f"  lineas con cliente_id nulo     : {n_cli_nulo:,}")
print(f"  lineas con cantidad <= 0       : {n_cant_0:,}")
print(f"  lineas con precio con coma     : {n_coma:,}")
print(f"  productos con color nulo       : {n_color:,}")
print(f"  productos con precio con signo $ : {n_dolar:,}")
display(p.groupBy("talla").count().orderBy("talla"))

Bronze · Generación